In [1]:
import re
from collections import defaultdict, Counter
import torch
from typing import List, Dict, Tuple

In [2]:
# 基础词汇表: 256字节 + 特殊token
token_to_id = {bytes([i]): i for i in range(256)}
token_to_id[b'</w>'] = 256
id_to_token = {v: k for k, v in token_to_id.items()}
merges = []  # 存储合并规则
special_tokens = {'<pad>': 257, '<unk>': 258, '<bos>': 259, '<eos>': 260}

token_to_id, id_to_token

({b'\x00': 0,
  b'\x01': 1,
  b'\x02': 2,
  b'\x03': 3,
  b'\x04': 4,
  b'\x05': 5,
  b'\x06': 6,
  b'\x07': 7,
  b'\x08': 8,
  b'\t': 9,
  b'\n': 10,
  b'\x0b': 11,
  b'\x0c': 12,
  b'\r': 13,
  b'\x0e': 14,
  b'\x0f': 15,
  b'\x10': 16,
  b'\x11': 17,
  b'\x12': 18,
  b'\x13': 19,
  b'\x14': 20,
  b'\x15': 21,
  b'\x16': 22,
  b'\x17': 23,
  b'\x18': 24,
  b'\x19': 25,
  b'\x1a': 26,
  b'\x1b': 27,
  b'\x1c': 28,
  b'\x1d': 29,
  b'\x1e': 30,
  b'\x1f': 31,
  b' ': 32,
  b'!': 33,
  b'"': 34,
  b'#': 35,
  b'$': 36,
  b'%': 37,
  b'&': 38,
  b"'": 39,
  b'(': 40,
  b')': 41,
  b'*': 42,
  b'+': 43,
  b',': 44,
  b'-': 45,
  b'.': 46,
  b'/': 47,
  b'0': 48,
  b'1': 49,
  b'2': 50,
  b'3': 51,
  b'4': 52,
  b'5': 53,
  b'6': 54,
  b'7': 55,
  b'8': 56,
  b'9': 57,
  b':': 58,
  b';': 59,
  b'<': 60,
  b'=': 61,
  b'>': 62,
  b'?': 63,
  b'@': 64,
  b'A': 65,
  b'B': 66,
  b'C': 67,
  b'D': 68,
  b'E': 69,
  b'F': 70,
  b'G': 71,
  b'H': 72,
  b'I': 73,
  b'J': 74,
  b'K': 75,
  b'L': 

In [4]:
a = "test"
a.encode('utf-8')

b'test'

In [11]:
bytes('a'.encode('utf-8'))

b'a'

In [25]:
bytes([97]), bytes(97)

(b'a',
 b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00')

In [3]:
def _add_special_tokens():
        """添加特殊token到词汇表"""
        for token, idx in special_tokens.items():
            token_to_id[token.encode('utf-8')] = idx
            id_to_token[idx] = token.encode('utf-8')

_add_special_tokens()

token_to_id, id_to_token

({b'\x00': 0,
  b'\x01': 1,
  b'\x02': 2,
  b'\x03': 3,
  b'\x04': 4,
  b'\x05': 5,
  b'\x06': 6,
  b'\x07': 7,
  b'\x08': 8,
  b'\t': 9,
  b'\n': 10,
  b'\x0b': 11,
  b'\x0c': 12,
  b'\r': 13,
  b'\x0e': 14,
  b'\x0f': 15,
  b'\x10': 16,
  b'\x11': 17,
  b'\x12': 18,
  b'\x13': 19,
  b'\x14': 20,
  b'\x15': 21,
  b'\x16': 22,
  b'\x17': 23,
  b'\x18': 24,
  b'\x19': 25,
  b'\x1a': 26,
  b'\x1b': 27,
  b'\x1c': 28,
  b'\x1d': 29,
  b'\x1e': 30,
  b'\x1f': 31,
  b' ': 32,
  b'!': 33,
  b'"': 34,
  b'#': 35,
  b'$': 36,
  b'%': 37,
  b'&': 38,
  b"'": 39,
  b'(': 40,
  b')': 41,
  b'*': 42,
  b'+': 43,
  b',': 44,
  b'-': 45,
  b'.': 46,
  b'/': 47,
  b'0': 48,
  b'1': 49,
  b'2': 50,
  b'3': 51,
  b'4': 52,
  b'5': 53,
  b'6': 54,
  b'7': 55,
  b'8': 56,
  b'9': 57,
  b':': 58,
  b';': 59,
  b'<': 60,
  b'=': 61,
  b'>': 62,
  b'?': 63,
  b'@': 64,
  b'A': 65,
  b'B': 66,
  b'C': 67,
  b'D': 68,
  b'E': 69,
  b'F': 70,
  b'G': 71,
  b'H': 72,
  b'I': 73,
  b'J': 74,
  b'K': 75,
  b'L': 

In [21]:
# text = "例如中文和English混合的场景"
text = "hello world"
words = re.split(r'(\s)', text.strip())  # 分割出单词和空格
word_freqs = Counter()  # 统计词频

word_freqs.update(w for w in words if w)  # 更新词频统计

words, word_freqs

(['hello', ' ', 'world'], Counter({'hello': 1, ' ': 1, 'world': 1}))

In [28]:
a = "🚀👍"
b = list(a.encode('utf-8'))
print(b)

for i in b:
    print(bytes([i]))

[240, 159, 154, 128, 240, 159, 145, 141]
b'\xf0'
b'\x9f'
b'\x9a'
b'\x80'
b'\xf0'
b'\x9f'
b'\x91'
b'\x8d'


In [26]:
"""初始化单词的分割状态"""
splits = {}
splits2 = {}
for word, freq in word_freqs.items():
    # 转换为字节并添加结束符
    byte_sequence = list(word.encode('utf-8'))
    print(byte_sequence)
    splits[word] = [bytes([b]) for b in byte_sequence] + [b'</w>']
    splits2[word] = [bytes([b]) for b in word.encode('utf-8')] + [b'</w>']

splits, splits2

[104, 101, 108, 108, 111]
[32]
[119, 111, 114, 108, 100]


({'hello': [b'h', b'e', b'l', b'l', b'o', b'</w>'],
  ' ': [b' ', b'</w>'],
  'world': [b'w', b'o', b'r', b'l', b'd', b'</w>']},
 {'hello': [b'h', b'e', b'l', b'l', b'o', b'</w>'],
  ' ': [b' ', b'</w>'],
  'world': [b'w', b'o', b'r', b'l', b'd', b'</w>']})